In [0]:
"""
Cuaderno 05: Generación del Dataset de Entrenamiento 
Propósito: Consolidar la Spine con el Feature Store, limpiar nulos, 
imputar valores y registrar metadatos finales en Unity Catalog.
"""

# 1. CONTROL DE DEPENDENCIAS
try:
    from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
    print("🚀 Usando FeatureEngineeringClient (Moderna)")
except ImportError:
    from databricks.feature_store import FeatureStoreClient as FeatureEngineeringClient
    from databricks.feature_store import FeatureLookup
    print("📦 Usando FeatureStoreClient (Clásica)")

from pyspark.sql.functions import col, max, count, when, round, current_timestamp
from datetime import datetime, timezone

fe = FeatureEngineeringClient()

# 2. CONFIGURACIÓN DE RUTAS Y TABLAS
catalog = "workspace"
schema = "abandono_carrito_comercio_electronico"

gold_abandonment_spine = f"{catalog}.{schema}.gold_abandonment_spine"
gold_user_profile = f"{catalog}.{schema}.gold_user_profile"
gold_user_behavior_features = f"{catalog}.{schema}.gold_user_behavior_features"
gold_training_dataset_table = f"{catalog}.{schema}.gold_abandonment_training_dataset"

# 3. FUNCIONES DE CONTROL DEL PROFESOR (Versionado y Fechas)
def _get_next_semantic_version(table_name):
    if not spark.catalog.tableExists(table_name): return 0
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {table_name}")
        row = props.filter("key = 'ml.delta_semantic_version'").first()
        return int(row["value"]) + 1 if row else 0
    except: return 0

def _get_current_max_date(table_name):
    if not spark.catalog.tableExists(table_name): return None
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {table_name}")
        row = props.filter("key = 'ml.data_max_date'").first()
        return row["value"] if row else None
    except: return None

###############################################################################
# 4. CARGA DE DATOS Y DEFINICIÓN DE LOOKUPS
###############################################################################

spine_df = spark.read.table(gold_abandonment_spine)
entity_key = "user_id"
timestamp_key = "event_timestamp"

# Características de Perfil (SCD2)
profile_lookup = FeatureLookup(
    table_name = gold_user_profile,
    lookup_key = entity_key,
    timestamp_lookup_key = timestamp_key 
)

# Características de Comportamiento (PiT)
behavior_lookup = FeatureLookup(
    table_name = gold_user_behavior_features,
    lookup_key = entity_key,
    timestamp_lookup_key = timestamp_key
)

# Creación del Training Set Lógico
training_set = fe.create_training_set(
    df = spine_df,
    feature_lookups = [profile_lookup, behavior_lookup],
    label = "will_abandon",
    exclude_columns = ["label_available_date", "ingestion_timestamp", "source_file"]
)

raw_training_df = training_set.load_df()

###############################################################################
# 5. CIRUGÍA DE LIMPIEZA E IMPUTACIÓN (Tu lógica de negocio)
###############################################################################

# A) Filtrar usuarios sin perfil
clean_training_df = raw_training_df.filter(col("age").isNotNull())

# B) Imputar nulos en agregaciones
agg_cols = [c for c in clean_training_df.columns if any(x in c for x in ["count", "sum", "avg", "max", "ratio"])]
clean_training_df = clean_training_df.fillna(0, subset=agg_cols)

# C) Imputar nulos en perfiles
clean_training_df = clean_training_df.fillna("Unknown", subset=["gender", "country", "preferred_device"])

###############################################################################
# 6. CONTROLES DE CALIDAD DEL PROFESOR
###############################################################################

print("--- CONTROLES DE CALIDAD ---")

# Control 1: Nulos en columnas de características
feature_columns = [c for c in clean_training_df.columns if c not in [entity_key, timestamp_key, "will_abandon"]]
null_counts = clean_training_df.select([count(when(col(c).isNull(), c)).alias(c) for c in feature_columns]).collect()[0]
print("\nNulos detectados:")
for c in feature_columns:
    print(f" - {c}: {null_counts[c]:,}")

# Control 2: Consistencia de filas (Spine vs Final)
spine_count = spine_df.count()
final_count = clean_training_df.count()
print(f"\nFilas en Spine: {spine_count:,}")
print(f"Filas en Dataset Final: {final_count:,} (Diferencia: {spine_count - final_count:,} por filtros)")

# Control 3: Balance de clases
print("\nBalance de clases:")
class_balance = clean_training_df.groupBy("will_abandon").count().withColumn("pct", round(col("count")/final_count*100, 2)).collect()
for row in class_balance:
    print(f" - Label {row['will_abandon']}: {row['count']:,} ({row['pct']}%)")

###############################################################################
# 7. PERSISTENCIA Y TRAZABILIDAD (MLOPS)
###############################################################################

# Obtener versiones antes de escribir
delta_semantic_version = _get_next_semantic_version(gold_training_dataset_table)
data_previous_max_date = _get_current_max_date(gold_training_dataset_table)

# Escritura física
(clean_training_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(gold_training_dataset_table))

# Capturar versión física y fecha máxima tras el WRITE
delta_physical_version = int(spark.sql(f"DESCRIBE HISTORY {gold_training_dataset_table}").select("version").first()[0])
data_max_date = clean_training_df.agg(max(col(timestamp_key))).collect()[0][0].strftime("%Y-%m-%d")
generated_at = datetime.now(timezone.utc).isoformat()

# REGISTRO DE TAGS (Para humanos en UI)
spark.sql(f"""
    ALTER TABLE {gold_training_dataset_table}
    SET TAGS (
        'delta_semantic_version' = '{delta_semantic_version}',
        'data_max_date' = '{data_max_date}',
        'label_col' = 'will_abandon',
        'project' = 'cart_abandonment_prediction',
        'num_rows' = '{final_count}'
    )
""")

# REGISTRO DE PROPERTIES (Para trazabilidad del modelo)
spark.sql(f"""
    ALTER TABLE {gold_training_dataset_table}
    SET TBLPROPERTIES (
        'ml.delta_semantic_version' = '{delta_semantic_version}',
        'ml.delta_physical_version' = '{delta_physical_version}',
        'ml.data_max_date' = '{data_max_date}',
        'ml.data_previous_max_date' = '{data_previous_max_date}',
        'ml.label_col' = 'will_abandon',
        'ml.project' = 'cart_abandonment_prediction',
        'ml.generated_at' = '{generated_at}'
    )
""")

spark.sql(f"COMMENT ON TABLE {gold_training_dataset_table} IS 'Dataset de entrenamiento final niquelado. Filtra usuarios sin perfil e imputa 0 en agregaciones.'")

print(f"\n--- PROCESO FINALIZADO ---")
print(f"Versión Semántica: {delta_semantic_version} | Física: {delta_physical_version}")
print(f"Fecha máxima de datos: {data_max_date}")

###############################################################################
# 8. VERIFICACIÓN DE TIME TRAVEL
###############################################################################
print("\nVerificando acceso histórico (Time Travel):")
df_past = spark.read.format("delta").option("versionAsOf", delta_physical_version).table(gold_training_dataset_table)
print(f"Datos recuperados correctamente de la versión física {delta_physical_version}: {df_past.count():,} filas.")

🚀 Usando FeatureEngineeringClient (Moderna)
--- CONTROLES DE CALIDAD ---

Nulos detectados:
 - session_id: 0
 - item_category: 0
 - cart_value: 0
 - event_type: 0
 - user_type: 0
 - age: 0
 - gender: 0
 - country: 0
 - preferred_device: 0
 - registration_date: 0
 - favourite_category: 0
 - return_rate: 0
 - has_app_installed: 0
 - email_opt_in: 0
 - push_opt_in: 0
 - __END_AT: 10,356,715
 - age_group: 0
 - loyalty_segment: 0
 - count_events_1h: 0
 - sum_cart_value_1h: 0
 - avg_cart_value_1h: 0
 - distinct_categories_1h: 0
 - count_sessions_24h: 0
 - sum_cart_value_24h: 0
 - avg_cart_value_24h: 0
 - max_cart_value_24h: 0
 - distinct_categories_24h: 0
 - count_add_to_cart_24h: 0
 - count_events_7d: 0
 - sum_cart_value_7d: 0
 - distinct_categories_7d: 0
 - count_events_30d: 0
 - sum_cart_value_30d: 0
 - avg_cart_value_30d: 0
 - num_abandoned_confirmed_30d: 0
 - cart_value_24h_vs_avg_30d_ratio: 0

Filas en Spine: 25,218,000
Filas en Dataset Final: 10,356,715 (Diferencia: 14,861,285 por fil